<a href="https://colab.research.google.com/github/ashroy23/rag-linear-algebra/blob/main/RAG_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain-community pypdf -q

In [ ]:
from google.colab import files
pdf=files.upload()

Saving trimmed la pdf II.pdf to trimmed la pdf II (1).pdf
Saving trimmed la pdf III.pdf to trimmed la pdf III (1).pdf
Saving trimmed la pdf IV.pdf to trimmed la pdf IV (1).pdf
Saving trimmed la pdf V.pdf to trimmed la pdf V (1).pdf
Saving trimmed la pdf.pdf to trimmed la pdf (1).pdf


In [ ]:
from langchain_community.document_loaders import PyPDFLoader

In [ ]:
all_documents=[]

for file_name in pdf.keys():
  loader=PyPDFLoader(file_name)
  docs=loader.load()
  all_documents.extend(docs)

print(len(all_documents))
print(repr(all_documents[0].page_content[:500]))

428
''


In [ ]:
for i in range(5):
    print("Page", i)
    print(repr(all_documents[i].page_content))
    print("-"*40)

Page 0
''
----------------------------------------
Page 1
'Linear Algebra and Its Applications\nFourth Edition\nGilbert Strang\ny\nx y z \x1e \x0c \nz\nAx b\x1e\nb\n0\nAy b\x1e\n0Az \x1e\n0'
----------------------------------------
Page 2
'Contents\nPreface iv\n1 Matrices and Gaussian Elimination 1\n1.1 Introduction . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 1\n1.2 The Geometry of Linear Equations . . . . . . . . . . . . . . . . . . . . 4\n1.3 An Example of Gaussian Elimination . . . . . . . . . . . . . . . . . . 13\n1.4 Matrix Notation and Matrix Multiplication . . . . . . . . . . . . . . . . 21\n1.5 Triangular Factors and Row Exchanges . . . . . . . . . . . . . . . . . 36\n1.6 Inverses and Transposes . . . . . . . . . . . . . . . . . . . . . . . . . . 50\n1.7 Special Matrices and Applications . . . . . . . . . . . . . . . . . . . . 66\nReview Exercises . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 72\n2 Vector Spaces 77\n2.1 Vector Spaces and Subspace

In [ ]:
filtered_documents= [
    doc for doc in all_documents
    if len (doc.page_content.strip())>100
]

print(f"Before:{len(all_documents)}")
print(f"After:{len(filtered_documents)}")

Before:428
After:417


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    separators=["\n\n","\n",". ",", ",""]
)

In [ ]:
chunks=text_splitter.split_documents(filtered_documents)

print(f"Total number of chunks:{len(chunks)}")
print(f"Average chunk size:{sum(len(c.page_content)for c in chunks)//len(chunks)}")
print("\nSample Chunk:\n")
print(chunks[10].page_content)
print("\nIt's metadata:")
print(chunks[10].metadata)

Total number of chunks:1152
Average chunk size:833

Sample Chunk:

You might think I am exaggerating to use the word “beautiful” for a basic course
in mathematics. Not at all. This subject begins with two vectors v and w, pointing in
different directions. The key step is to take their linear combinations . We multiply to
get 3v and 4w, and we add to get the particular combination 3v + 4w. That new vector
is in the same plane as v and w. When we take all combinations, we are ﬁlling in the
whole plane. If I draw v and w on this page, their combinations cv + dw ﬁll the page
(and beyond), but they don’t go up from the page.
In the language of linear equations, I can solve cv + dw = b exactly when the vector
b lies in the same plane as v and w.
iv

It's metadata:
{'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20260426001905', 'source': 'trimmed la pdf II (1).pdf', 'total_pages': 80, 'page': 5, 'page_label': '6'}


In [ ]:
!pip install faiss-cpu langchain-huggingface sentence-transformers -q

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model=HuggingFaceEmbeddings (
    model_name="all-MiniLM-L6-v2"
)
print("\nEmbedding model loaded")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Embedding model loaded


In [ ]:
from langchain_community.vectorstores import FAISS

print ("Creating vector store....please wait")

vector_store=FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

print("Done!")
print (f"Total vectors stored:{vector_store.index.ntotal}")

Creating vector store....please wait
Done!
Total vectors stored:1152


In [ ]:
query="What are determinants?"
results= vector_store.similarity_search_with_score(query,k=5)

for i, (doc,score) in enumerate (results):
  print(f"\nResults: {i+1}")
  print(f"Score: {score:.4f}")
  print(f"\nSource:{doc.metadata['source']}, Page: {doc.metadata['page']}")
  print(doc.page_content[:300])



Results: 1
Score: 0.7737

Source:trimmed la pdf III (1).pdf, Page: 38
For this determinant is defined by
(2)
For by
(3a)
or
(3b)
Here, 
and is a determinant of order namely, the determinant of the submatrix of A
obtained from Aby omitting the row and column of the entry , that is, the jth row and
the kth column.
In this way, Dis defined in terms of ndeterminants of or

Results: 2
Score: 0.8615

Source:trimmed la pdf IV (1).pdf, Page: 82
It is easiest to give a diﬀerent deﬁnition of the determinant which is clearly well deﬁned
and then prove the earlier one in terms of Laplace expansion. Let ( i1, · · · , in) be an ordered
list of numbers from {1, · · · , n} . This means the order is important so (1, 2, 3) and (2, 1, 3)
are diﬀerent

Results: 3
Score: 0.8650

Source:trimmed la pdf IV (1).pdf, Page: 8
unusual.
This book features an ugly, elementary, and complete treatment of determinants early
in the book. Thus it might be considered as Linear algebra done wrong. I have done this
becaus

In [ ]:
!pip install langchain-google-genai -q

In [ ]:
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_API_KEY")
print("Key loaded:", bool(os.environ.get("GOOGLE_API_KEY")))

Key loaded: True


In [ ]:
import google.generativeai as genai
import os

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.5-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-preview-10-2025
models/antigravity-preview-05-2026
models/

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage

# Step 1 - set up Gemini
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0
)

# Step 2 - your question
query = "What are determinants?"

# Step 3 - retrieve relevant chunks
docs = vector_store.similarity_search(query, k=3)

# Step 4 - join chunks into one block of text
context = "\n\n".join([doc.page_content for doc in docs])

# Step 5 - build the prompt
messages = [
    SystemMessage(content="You are a helpful assistant that answers questions about linear algebra. Answer ONLY using the context provided. If the answer is not in the context say I cannot find this in the provided material."),
    HumanMessage(content=f"Context:\n{context}\n\nQuestion: {query}\n\nAnswer clearly based only on the context above.")
]

# Step 6 - call Gemini and print the answer
response = llm.invoke(messages)
print(response.content)

Determinants are defined in relation to matrices. The context provides a recursive definition for determinants, starting with second-order determinants. A second-order determinant is defined as the entry itself when the submatrix consists of a single entry. For higher-order determinants, they are defined in terms of determinants of smaller order. Specifically, a determinant of order $n$ can be expanded by any row or column, using entries and determinants of order $n-1$. The context also mentions an alternative definition of the determinant using ordered lists of numbers from $\{1, \dots, n\}$.


In [ ]:
def ask(question, k=3):
    # Step 1 - retrieve relevant chunks
    docs = vector_store.similarity_search(question, k=k)

    # Step 2 - join chunks into context
    context = "\n\n".join([doc.page_content for doc in docs])

    # Step 3 - build prompt
    messages = [
        SystemMessage(content="You are a helpful assistant that answers questions about linear algebra. Answer ONLY using the context provided. If the answer is not in the context say I cannot find this in the provided material."),
        HumanMessage(content=f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer clearly based only on the context above.")
    ]

    # Step 4 - call Gemini
    response = llm.invoke(messages)
    return response.content

print("ask() function ready!")

ask() function ready!


In [ ]:
print(ask("What is an eigenvalue?"))
print(ask("How do you compute a determinant?"))
print(ask("What is Gaussian elimination?"))

An eigenvalue is a value for which equation (1) has a solution. Another term for it is a latent root.
The context provides several methods for computing determinants:

*   **Laplace Expansion:** While difficult for large matrices (requiring n! terms for an n x n matrix), it is a method for finding determinants.
*   **Elementary Row Operations:** Applying elementary row operations to a determinant can transform it into an upper triangular determinant, whose value is the product of its diagonal entries. It is important to note that interchanging two rows introduces a multiplicative factor of -1.
*   **Triangular Matrices:** The determinant of a triangular matrix is the product of its diagonal entries.
Gaussian elimination is a method used to solve systems of linear equations. It starts by subtracting multiples of the first equation from the other equations to eliminate a variable from those equations. This process can be performed on the augmented matrix of the system. The goal is to tra

In [ ]:
import time

# Your test questions
test_questions = [
    "What is an eigenvalue?",
    "What is Gaussian elimination?",
    "How do you compute a determinant?",
    "What is a vector space?",
    "What is linear independence?",
    "What is a matrix inverse?",
    "What is a dot product?",
    "What is an eigenvector?",
    "What is a non-invertible matrix with zero determinant?",
    "What is the rank of a matrix?"
]

# Loop through each question with a 30 second pause between each
for question in test_questions:
    print(f"\nQ: {question}")
    try:
        print(f"A: {ask(question)}")
    except Exception as e:
        print(f"Error: {e}")
    print("-" * 60)
    time.sleep(60)

print("\nDone!")


Q: What is an eigenvalue?
Error: Error calling model 'gemini-2.5-flash-lite' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite\nPlease retry in 45.257039834s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjec

In [ ]:
vector_store.save_local("faiss_index")
print("Saved!")

Saved!
